In [6]:
from scipy.stats import norm
import numpy as np
import time
import import_ipynb
import matplotlib.pyplot as plt
from scipy.stats import qmc

In [1]:
def pathwise_delta(S0, K, r, sigma, T, n_paths, type="call"):
    Z = np.random.normal(0, 1, n_paths)
    S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    
    if type == "call":
        indicator = (S_T > K).astype(float)
    else:
        indicator = (S_T < K).astype(float)
        # for a put, d(payoff)/dS_T = -1{S_T < K}, so overall sign flips
    
    sign = 1 if type == "call" else -1
    delta_paths = np.exp(-r*T) * sign * indicator * (S_T / S0)
    
    delta = delta_paths.mean()
    se = delta_paths.std(ddof=1) / np.sqrt(n_paths)
    return delta, se

In [2]:
def pathwise_vega(S0, K, r, sigma, T, n_paths, type="call"):
    Z = np.random.normal(0, 1, n_paths)
    S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    
    if type == "call":
        indicator = (S_T > K).astype(float)
    else:
        indicator = (S_T < K).astype(float)
    
    dS_T_dsigma = S_T * (np.sqrt(T)*Z - sigma*T)
    vega_paths = np.exp(-r*T) * indicator * dS_T_dsigma
    
    vega = vega_paths.mean()
    se = vega_paths.std(ddof=1) / np.sqrt(n_paths)
    return vega, se

In [3]:
def lr_delta(S0, K, r, sigma, T, n_paths, type="call"):
    Z = np.random.normal(0, 1, n_paths)
    S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    
    if type == "call":
        payoff = np.maximum(S_T - K, 0)
    else:
        payoff = np.maximum(K - S_T, 0)
    
    score = Z / (S0 * sigma * np.sqrt(T))
    delta_paths = np.exp(-r*T) * payoff * score
    
    delta = delta_paths.mean()
    se = delta_paths.std(ddof=1) / np.sqrt(n_paths)
    return delta, se

In [4]:
def lr_vega(S0, K, r, sigma, T, n_paths, type="call"):
    Z = np.random.normal(0, 1, n_paths)
    S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    
    if type == "call":
        payoff = np.maximum(S_T - K, 0)
    else:
        payoff = np.maximum(K - S_T, 0)
    
    score = (Z**2 - 1)/sigma - Z*np.sqrt(T)
    vega_paths = np.exp(-r*T) * payoff * score
    
    vega = vega_paths.mean()
    se = vega_paths.std(ddof=1) / np.sqrt(n_paths)
    return vega, se

In [5]:
def fd_delta(S0, K, r, sigma, T, n_paths, type="call", h=0.01):
    seed = 42
    rng1 = np.random.default_rng(seed)
    Z = rng1.standard_normal(n_paths)
    
    S_T_up = (S0+h) * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    S_T_down = (S0-h) * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    
    if type == "call":
        payoff_up = np.maximum(S_T_up - K, 0)
        payoff_down = np.maximum(S_T_down - K, 0)
    else:
        payoff_up = np.maximum(K - S_T_up, 0)
        payoff_down = np.maximum(K - S_T_down, 0)
    
    price_up = np.exp(-r*T) * payoff_up.mean()
    price_down = np.exp(-r*T) * payoff_down.mean()
    
    delta = (price_up - price_down) / (2*h)
    return delta

def fd_vega(S0, K, r, sigma, T, n_paths, type="call", h=0.01):
    seed = 42
    rng1 = np.random.default_rng(seed)
    Z = rng1.standard_normal(n_paths)
    
    S_T_up = S0 * np.exp((r - 0.5*(sigma+h)**2)*T + (sigma+h)*np.sqrt(T)*Z)
    S_T_down = S0 * np.exp((r - 0.5*(sigma-h)**2)*T + (sigma-h)*np.sqrt(T)*Z)
    
    if type == "call":
        payoff_up = np.maximum(S_T_up - K, 0)
        payoff_down = np.maximum(S_T_down - K, 0)
    else:
        payoff_up = np.maximum(K - S_T_up, 0)
        payoff_down = np.maximum(K - S_T_down, 0)
    
    price_up = np.exp(-r*T) * payoff_up.mean()
    price_down = np.exp(-r*T) * payoff_down.mean()
    
    vega = (price_up - price_down) / (2*h)
    return vega